In [1]:
import os
import numpy as np 
import pandas as pd 
import cv2 as cv
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,MaxPooling2D,Activation,Flatten,Conv2D,BatchNormalization,Dropout
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras import layers
from tensorflow.keras import Model
from tensorflow import keras



test_data = pd.read_csv("../input/petfinder-pawpularity-score/test.csv").to_numpy()
train_data = pd.read_csv("../input/petfinder-pawpularity-score/train.csv").to_numpy()

TEST_PATH = '../input/petfinder-pawpularity-score/test'
TRAIN_PATH = '../input/petfinder-pawpularity-score/train'

dimensions = (128, 128, 1)


2026-01-07 01:19:03.085730: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767748743.098567      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767748743.102520      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-07 01:19:03.118280: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
class DataGenerator(keras.utils.Sequence):
    def __init__(self, directory, list_IDs, n_channels=3, batch_size=8, dim=dimensions, shuffle=True):
        'Initialization'
        self.dim = dim
        self.directory = directory
        self.batch_size = batch_size
        self.list_IDs = list_IDs
        self.shuffle = shuffle
        self.on_epoch_end()
        
    def __len__(self):
        return int(np.floor(len(self.list_IDs) / self.batch_size))
        
    def on_epoch_end(self):
        self.indexes = np.arange(len(self.list_IDs))
        if self.shuffle == True:
            np.random.shuffle(self.indexes)
            
    def __data_generation(self, list_IDs_temp):
        
        X = np.empty((self.batch_size, *self.dim))
        y = []
            
        for batch_number, id_temp_entry in enumerate(list_IDs_temp):
            path = self.directory + '/' + id_temp_entry[0] + '.jpg'
                
            image = cv.imread(path, cv.IMREAD_GRAYSCALE)
            image = cv.resize(image, (self.dim[0],self.dim[1]))
            image = image.reshape(dimensions)
            
            X[batch_number] = image
            
            if len(id_temp_entry) >= 14:
                y.append(id_temp_entry[13])
            
        y = np.array(y)
        if len(y) == 0:
            return X, []
        
        
        return X,  y
                     
    def __getitem__(self, index):
            
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]

            
        list_IDs_temp = [self.list_IDs[k] for k in indexes]

            
        X, y = self.__data_generation(list_IDs_temp)
        
        if len(y) == 0:
            return X

        return X, y


In [3]:
model = Sequential()


model.add(keras.Input(shape=dimensions))
# model.add(keras.layers.RandomRotation(
#     0.15, fill_mode='reflect', interpolation='bilinear',
#     seed=None, fill_value=0.0
# )
# )


model.add(Conv2D(filters=128,padding="same",kernel_size=(15,15)))
model.add(Activation('relu'))
model.add(MaxPooling2D(3,3))


model.add(Conv2D(filters=128,padding="same",kernel_size=(11,11)))
model.add(Activation('relu'))
model.add(MaxPooling2D(3,3))



model.add(Flatten())
model.add(Dense(16))
model.add(Activation("relu"))

model.add(Dropout(0.45))

model.add(Flatten())
model.add(Dense(16))
model.add(Activation("relu"))


# model.add(Flatten())
# model.add(Dense(512))
# model.add(Activation("relu"))

# model.add(Flatten())
# model.add(Dense(256))
# model.add(Activation("relu"))


# model.add(Flatten())
# model.add(Dense(128))
# model.add(Activation("relu"))

# model.add(Flatten())
# model.add(Dense(64))
# model.add(Activation("relu"))


# model.add(Flatten())
# model.add(Dense(32))
# model.add(Activation("relu"))

# model.add(Flatten())
# model.add(Dense(16))
# model.add(Activation("relu"))

# model.add(Flatten())
# model.add(Dense(8))
# model.add(Activation("relu"))


model.add(Flatten())
model.add(Dense(1))

lr_schedule = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-4,
    decay_steps=100000,
    decay_rate=1.2,
    staircase=True)

opt = SGD(learning_rate=lr_schedule)
loss_fn = keras.losses.MeanAbsoluteError()
model.compile(loss=loss_fn,optimizer=opt,metrics=['mae', 'mse'])


2026-01-07 01:19:10.007111: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1767748750.007558      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 40095 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:85:00.0, compute capability: 8.6


In [4]:
training_generator = DataGenerator(TRAIN_PATH, train_data, batch_size=64)
validation_generator = DataGenerator(TEST_PATH, test_data, batch_size=1)

model.fit(training_generator,
                    validation_data=validation_generator, epochs = 10)
model.summary()
results = model.predict(validation_generator)


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1767748751.367018      65 service.cc:148] XLA service 0x7f7480004b40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1767748751.367043      65 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-01-07 01:19:11.383352: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1767748751.450386      65 cuda_dnn.cc:529] Loaded cuDNN version 90300


  1/139 ━━━━━━━━━━━━━━━━━━━━ 14:53 6s/step - loss: 42.9154 - mae: 42.9154 - mse: 2515.7646

I0000 00:00:1767748757.308603      65 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step - loss: 24.1310 - mae: 24.1310 - mse: 997.8629

ValueError: None values not supported.

In [5]:
submission = []

for index, result in enumerate(results):
    submission.append([test_data[index][0], float(result[0])])
    
    
df = pd.DataFrame(submission, columns=['Id', 'Pawpularity'])
df = df.sort_values('Id')
df.to_csv('submission.csv', index=False)
df


NameError: name 'results' is not defined